# UCI HAR — ICE & SEM on real 3-D smartphone-accelerometer data

This notebook runs the unsupervised estimators of `awesomePMC` — **ICE** (PR1) and **SEM** (PR2) — on the *Human Activity Recognition Using Smartphones* dataset (UCI ID 240). It demonstrates the package on **multivariate observations (d = 3)** from a **real-world** sensor, complementing the synthetic-data quickstart.

**Goal.** Segment a long 3-D accelerometer recording into two contrasted activity regimes — `WALKING` (high movement) versus `LAYING` (no movement) — using an HMC-IN K = 2 model with a multivariate-Gaussian margin per state.

**Network-friendly.** The dataset is downloaded on first run and cached in `data/uci_har/`. If the download fails or is declined, the notebook falls back transparently to a synthetic 3-D signal simulated from the bundled `hmc_in_mvn_k2_d3.toml` fixture — the rest of the notebook is identical.

**Reference.**  D. Anguita *et al.*, *A Public Domain Dataset for Human Activity Recognition Using Smartphones*, ESANN 2013.

In [ ]:
from __future__ import annotations

import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pmcprg.pmc import PMCModel, classify, error_rate, ice, sem, simulate

## 1. Load the data — UCI HAR if available, synthetic fallback otherwise

The UCI HAR archive bundles 30 subjects × 6 activities of 50 Hz accelerometer + gyroscope recordings, pre-windowed into 2.56 s segments. We pick one subject (`subject_id = 1`) and stitch their `WALKING` (label 1) and `LAYING` (label 6) windows into a single 3-D time series. The result is roughly 3 000 samples of total-acceleration `(x, y, z)`.

Set `ALLOW_DOWNLOAD = False` below to force the synthetic fallback.

In [ ]:
ALLOW_DOWNLOAD = True
DATA_DIR       = Path("data/uci_har")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_URL    = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human+activity+recognition+using+smartphones.zip"
)
EXTRACT_ROOT   = DATA_DIR / "UCI HAR Dataset"

# Activity codes — see UCI HAR Dataset/activity_labels.txt
ACTIVITY_WALKING = 1
ACTIVITY_LAYING  = 6
SUBJECT_ID       = 1


def fetch_uci_har_zip(url: str, dest_dir: Path, timeout: int = 30) -> bool:
    """Download and extract the UCI HAR zip if the cache is empty.

    Returns ``True`` if the extracted dataset is present at the end, else
    ``False`` (network error, archive layout unexpected, …).
    """
    if (dest_dir / "UCI HAR Dataset" / "train" / "y_train.txt").exists():
        print(f"\u2713 cache present at {dest_dir / 'UCI HAR Dataset'} — skipping download")
        return True
    try:
        print(f"Downloading {url} \u2026  (~60 MB)")
        with urllib.request.urlopen(url, timeout=timeout) as r:
            raw = r.read()
        print(f"  unpacking \u2192 {dest_dir}")
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            zf.extractall(dest_dir)
    except (urllib.error.URLError, TimeoutError, zipfile.BadZipFile, OSError) as exc:
        print(f"  download failed: {exc!r}")
        return False
    return (dest_dir / "UCI HAR Dataset" / "train" / "y_train.txt").exists()


def load_uci_har_subset(
    root: Path, subject_id: int, activities: tuple[int, int],
) -> tuple[np.ndarray, np.ndarray]:
    """Stitch the windows of one subject for two given activities.

    Returns
    -------
    X_ref : (N,) int   — ground-truth class labels in {0, 1}
    Y     : (N, 3) float — total-acceleration time series (x, y, z)
    """
    train_dir = root / "train"
    sig_dir   = train_dir / "Inertial Signals"
    subj  = np.loadtxt(train_dir / "subject_train.txt", dtype=int)
    y_act = np.loadtxt(train_dir / "y_train.txt",       dtype=int)
    acc_x = np.loadtxt(sig_dir   / "total_acc_x_train.txt")
    acc_y = np.loadtxt(sig_dir   / "total_acc_y_train.txt")
    acc_z = np.loadtxt(sig_dir   / "total_acc_z_train.txt")
    # Pick windows for this subject in the two requested activities, in
    # chronological (file) order. Each window is 128 samples \u2014 we keep the
    # first 64 of each to avoid the 50 % overlap inherent to UCI HAR.
    mask = (subj == subject_id) & np.isin(y_act, activities)
    if not mask.any():
        raise RuntimeError(
            f"no windows found for subject={subject_id} "
            f"activities={activities} \u2014 unexpected dataset layout?"
        )
    idx     = np.where(mask)[0]
    labels  = (y_act[idx] == activities[1]).astype(int)   # 0 = first, 1 = second
    half    = 64
    windows = np.stack([acc_x[idx, :half],
                        acc_y[idx, :half],
                        acc_z[idx, :half]], axis=-1)        # (n_win, 64, 3)
    Y      = windows.reshape(-1, 3)                         # (N, 3)
    X_ref  = np.repeat(labels, half).astype(int)             # (N,)
    return X_ref, Y


use_real = ALLOW_DOWNLOAD and fetch_uci_har_zip(DATASET_URL, DATA_DIR)
if use_real:
    X_ref, Y = load_uci_har_subset(
        EXTRACT_ROOT, SUBJECT_ID, (ACTIVITY_WALKING, ACTIVITY_LAYING),
    )
    DATA_SOURCE = f"UCI HAR (subject {SUBJECT_ID}, WALKING vs LAYING)"
else:
    print("Falling back to synthetic 3-D signal from hmc_in_mvn_k2_d3.toml\u2026")
    fixture  = PMCModel("pmcprg/pmc/models/hmc_in_mvn_k2_d3.toml")
    X_ref, Y = simulate(fixture, N=2000, seed=0)
    DATA_SOURCE = "synthetic (HMC-IN K=2, d=3, multivariate Gaussian)"

print(f"Data source: {DATA_SOURCE}")
print(f"  N = {len(Y)}   d = {Y.shape[1]}")
print(f"  class counts: 0\u2192{int((X_ref == 0).sum())}  1\u2192{int((X_ref == 1).sum())}")

## 2. Visualise the 3-D signal and the ground-truth labels

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True)
for k, ax in enumerate(axes[:3]):
    ax.plot(Y[:, k], lw=0.6)
    ax.set_ylabel("x y z"[k])
axes[3].plot(X_ref, drawstyle="steps-post", lw=0.8, color="black")
axes[3].set_ylabel("class")
axes[3].set_xlabel("time-step n")
axes[0].set_title(f"3-D accelerometer + ground truth — {DATA_SOURCE}")
fig.tight_layout()
plt.show()

## 3. Build the initial model — HMC-IN K = 2, d = 3 multivariate Gaussian

We start from the bundled `hmc_in_mvn_k2_d3.toml` fixture. Its initial parameters are not particularly close to the data: we will let the K-means warm-start fix that.

In [ ]:
init_mdl = PMCModel("pmcprg/pmc/models/hmc_in_mvn_k2_d3.toml")
print(f"Initial model: variant={init_mdl.variant.value}, K={init_mdl.K}, d={init_mdl.d}")
for blk in init_mdl.raw["margins"]:
    print(f"  state {blk['i']}: dist={blk['dist']}  mean={blk['params']['mean']}")

## 4. ICE estimation — K-means warm-start, refit margins

Iterative Conditional Estimation. Deterministic, monotone log-likelihood under standard conditions, well-suited as a starting baseline.

In [ ]:
ice_cfg = {
    "init":         "kmeans",
    "kmeans_seed":  0,
    "fit_margins":  True,
    "max_iter":     15,
    "candidates":   [],          # K-means + multivariate Gaussian: no copula candidates
}
fitted_ice, trace_ice = ice(init_mdl, Y, ice_cfg=ice_cfg)
X_hat_ice, _, _       = classify(fitted_ice, Y)
er_ice                = error_rate(X_ref, X_hat_ice)
print(f"ICE  iters={len(trace_ice.log_liks):2d}  final LL={trace_ice.log_liks[-1]:.2f}  error={er_ice * 100:.2f} %")

## 5. SEM estimation — same warm-start, stochastic completion

Stochastic EM (PR2). Draws `X̃ ~ P(X | Y)` at each iteration via Forward-Filter Backward-Sample and runs the same M-step on the hard labels. Log-likelihood fluctuates around its stationary regime instead of converging deterministically.

In [ ]:
sem_cfg = {
    "init":         "kmeans",
    "kmeans_seed":  0,
    "fit_margins":  True,
    "max_iter":     15,
    "sem_seed":     0,
    "candidates":   [],
}
fitted_sem, trace_sem = sem(init_mdl, Y, sem_cfg=sem_cfg)
X_hat_sem, _, _       = classify(fitted_sem, Y)
er_sem                = error_rate(X_ref, X_hat_sem)
print(f"SEM  iters={len(trace_sem.log_liks):2d}  final LL={trace_sem.log_liks[-1]:.2f}  error={er_sem * 100:.2f} %")

## 6. Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=False)

axes[0].plot(trace_ice.log_liks, marker="o", ms=4, label="ICE (deterministic)")
axes[0].plot(trace_sem.log_liks, marker="s", ms=4, alpha=0.75, label="SEM (stochastic)")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("log-likelihood")
axes[0].set_title(f"ICE vs SEM — {DATA_SOURCE}")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

axes[1].plot(X_ref,     drawstyle="steps-post", lw=0.9, label="truth")
axes[1].plot(X_hat_ice, drawstyle="steps-post", lw=0.6, alpha=0.7,
             label=f"ICE  (err={er_ice * 100:.1f} %)")
axes[1].plot(X_hat_sem, drawstyle="steps-post", lw=0.6, alpha=0.7,
             label=f"SEM  (err={er_sem * 100:.1f} %)")
axes[1].set_ylabel("class")
axes[1].set_xlabel("time-step n")
axes[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

## 7. Multivariate goodness-of-fit — `mks_2samp` (PR3)

Use the multivariate KS test to compare the fitted-model's *simulated* observations with the actual data, restricted to one class. If the fit is reasonable, the test should **not** reject the null hypothesis.

In [ ]:
from pmcprg.diagnostics import mks_2samp

# Sample synthetic data from the ICE-fitted model. We keep BOTH the true
# latent states X_sim and the observations Y_sim so the class-0 masks below
# select consistent rows on each side — masking Y_sim with X_hat_ice (the
# real-data classification) would slice unrelated rows because they only
# happen to share length.
X_sim, Y_sim = simulate(fitted_ice, N=len(Y), seed=1)
Y_obs_c0     = Y[X_ref == 0]
Y_sim_c0     = Y_sim[X_sim == 0]
# Subsample to keep the test fast (mks_2samp is O(N²)).
rng        = np.random.default_rng(0)
n_compare  = min(300, len(Y_obs_c0), len(Y_sim_c0))
Y_obs_sub  = Y_obs_c0[rng.choice(len(Y_obs_c0), n_compare, replace=False)]
Y_sim_sub  = Y_sim_c0[rng.choice(len(Y_sim_c0), n_compare, replace=False)]
res        = mks_2samp(Y_obs_sub, Y_sim_sub, alpha=0.05)
print(
    f"MKS 2-sample (class 0)  N={n_compare}  d=3  \n"
    f"  statistic      = {res.statistic:.4f}\n"
    f"  critical value = {res.critical_value:.4f}  (α={res.alpha})\n"
    f"  reject H0      = {res.reject}"
)

## Summary

This notebook ties together everything added by the four migration PRs:

| PR | Feature | Used here |
|---|---|---|
| 1 | K-means warm-start for ICE | `ice_cfg = {"init": "kmeans", \u2026}` |
| 2 | SEM (Stochastic EM) estimator | `from pmcprg.pmc import sem` |
| 3 | Multivariate KS goodness-of-fit | `from pmcprg.diagnostics import mks_2samp` |
| 4 | UCI HAR real-data benchmark | *this notebook* |

The data path stays *fully reproducible*: when offline or in CI the notebook falls back to the bundled multivariate fixture, and the rest runs unchanged.